# LeetCode 3768 — Minimum Inversion Count in Subarrays of Fixed Length bằng Python

- Link: https://leetcode.com/problems/minimum-inversion-count-in-subarrays-of-fixed-length/
- Độ khó: **Hard**
- Chủ đề: Cửa sổ trượt, Fenwick Tree (BIT), nén toạ độ, đếm nghịch thế, merge sort

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minInversionCount(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Đếm **nghịch thế** (cặp `i < j`, `nums[i] > nums[j]`) bằng merge sort và bằng Fenwick Tree.
2. Cập nhật số nghịch thế của cửa sổ **trượt** trong `O(log n)`:
   - thêm phần tử `x` vào **bên phải**: cộng số phần tử trong cửa sổ **lớn hơn** `x`;
   - bỏ phần tử `y` ở **bên trái**: trừ số phần tử còn lại **nhỏ hơn** `y`.
3. **Nén toạ độ** để dùng BIT với giá trị tới `10^9`.
4. Xử lý đúng phần tử **bằng nhau** (không phải nghịch thế).
5. So sánh 4 cách: vét cạn `O(n·k²)`, merge sort mỗi cửa sổ, `insort`, BIT trượt `O(n log n)`.

## 2. Đề bài & ví dụ

Cho `nums` độ dài `n` và `k`. Nghịch thế là cặp `(i, j)`, `i < j`, `nums[i] > nums[j]`. Trả về **số nghịch thế nhỏ nhất** trong mọi mảng con độ dài `k`.

**Ví dụ 1:** `nums = [3,1,2,5,4], k = 3` → `0` (cửa sổ `[1,2,5]`).

**Ví dụ 2:** `nums = [5,3,2,1], k = 4` → `6`.

**Ví dụ 3:** `nums = [2,1], k = 1` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^9`, `1 ≤ k ≤ n`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right, insort
import random, time

examples = [([3, 1, 2, 5, 4], 3, 0), ([5, 3, 2, 1], 4, 6), ([2, 1], 1, 0)]
for a, k, exp in examples:
    print(a, k, "-> kỳ vọng", exp)

[3, 1, 2, 5, 4] 3 -> kỳ vọng 0
[5, 3, 2, 1] 4 -> kỳ vọng 6
[2, 1] 1 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Đếm nghịch thế bằng merge sort

Khi trộn hai nửa đã sắp xếp `L` và `R`, mỗi lần lấy phần tử từ `R` trước các phần tử còn lại của `L` (do `L[i] > R[j]`), ta cộng `len(L) − i` nghịch thế. Tổng `O(m log m)`.

In [2]:
def inv_merge(a):
    if len(a) <= 1:
        return a[:], 0
    m = len(a) // 2
    L, x = inv_merge(a[:m]); R, y = inv_merge(a[m:])
    res, i, j, c = [], 0, 0, x + y
    while i < len(L) and j < len(R):
        if L[i] <= R[j]:                    # bằng nhau KHÔNG phải nghịch thế
            res.append(L[i]); i += 1
        else:
            res.append(R[j]); j += 1; c += len(L) - i
    res += L[i:] + R[j:]
    return res, c

print(inv_merge([3, 1, 2])[1], inv_merge([5, 3, 2, 1])[1], inv_merge([2, 2, 1])[1])

2 6 2


### 3.2. Fenwick Tree đếm theo giá trị

Lưu `cnt[v]` = số phần tử có hạng (rank) `v` trong cửa sổ. `prefix(v)` = số phần tử có hạng `≤ v`. Khi đó:

- số phần tử `> x` = `size − prefix(rank(x))`;
- số phần tử `< y` = `prefix(rank(y) − 1)`.

In [3]:
class BIT:
    def __init__(self, n):
        self.n = n; self.c = [0] * (n + 1)
    def add(self, i, d):
        while i <= self.n:
            self.c[i] += d; i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.c[i]; i -= i & -i
        return s

b = BIT(5)
for r in [3, 1, 2]: b.add(r, 1)
print("số phần tử > hạng 1:", 3 - b.prefix(1), "| số phần tử < hạng 3:", b.prefix(2))

số phần tử > hạng 1: 2 | số phần tử < hạng 3: 2


### 3.3. Nén toạ độ

Giá trị tới `10^9` nhưng chỉ có tối đa `n` giá trị khác nhau ⇒ ánh xạ mỗi giá trị sang hạng `1..m` theo thứ tự tăng (giữ thứ tự so sánh, giữ quan hệ bằng nhau).

In [4]:
vals = [1000000000, 7, 7, 42]
sorted_unique = sorted(set(vals))
rank = {v: i + 1 for i, v in enumerate(sorted_unique)}
print([rank[v] for v in vals])

[3, 1, 1, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hai cửa sổ liên tiếp `[i−1 .. i+k−2]` và `[i .. i+k−1]` chỉ khác nhau **một phần tử ở mỗi đầu**:

- Bỏ `y = nums[i−1]` ở đầu trái: `y` đứng trước mọi phần tử còn lại, nên nó góp đúng `#{z trong cửa sổ : z < y}` nghịch thế ⇒ trừ đi.
- Thêm `x = nums[i+k−1]` ở đầu phải: `x` đứng sau mọi phần tử, góp `#{z : z > x}` ⇒ cộng vào.

Ví dụ 1: cửa sổ `[3,1,2]` có 2. Bỏ `3` (2 phần tử nhỏ hơn) ⇒ 0; thêm `5` (0 phần tử lớn hơn) ⇒ 0 cho `[1,2,5]`. Bỏ `1` (0) ⇒ 0; thêm `4` (1 phần tử `5` lớn hơn) ⇒ 1 cho `[2,5,4]`. Tối thiểu = 0. ✔

## 5. Cách 1 — Brute Force: đếm từng cửa sổ bằng hai vòng lặp

In [5]:
def brute(nums, k):
    best = None
    for s in range(len(nums) - k + 1):
        w = nums[s:s + k]
        c = sum(1 for i in range(k) for j in range(i + 1, k) if w[i] > w[j])
        best = c if best is None else min(best, c)
    return best

for a, k, exp in examples:
    print(brute(a, k), "kỳ vọng", exp)

0 kỳ vọng 0
6 kỳ vọng 6
0 kỳ vọng 0


**Nút thắt:** `O((n − k + 1) · k²)`, với `k ≈ n/2 = 5·10^4` là khoảng `10^14` phép toán.

## 6. Ý tưởng thuật toán / bước đột phá

**Cửa sổ trượt + BIT theo hạng:**

1. Nén `nums` thành hạng `1..m`.
2. Dựng cửa sổ đầu: thêm lần lượt `k` phần tử, mỗi lần cộng `#{> x}` (vì phần tử mới luôn ở bên phải).
3. Trượt: trừ `#{< y}` của phần tử bỏ đi (sau khi xoá `y` khỏi BIT hoặc trước — `y` không tự so với chính nó vì so sánh nghiêm ngặt), rồi cộng `#{> x}` của phần tử mới.
4. Lấy `min` qua mọi cửa sổ.

Mỗi bước `O(log m)` ⇒ tổng `O(n log n)`. Số nghịch thế tối đa `k(k−1)/2 ≈ 5·10^9` — Python không lo tràn số.

## 7. Các cách tiếp theo

- **Cách 2:** merge sort trên từng cửa sổ — `O((n−k+1) · k log k)`.
- **Cách 3:** cửa sổ trượt + `bisect.insort` trên list đã sắp — đếm `O(log k)`, chèn/xoá `O(k)` ⇒ `O(n·k)` (thực tế nhanh nhờ `memmove`).
- **Cách 4:** cửa sổ trượt + Fenwick Tree — `O(n log n)` (lời giải nộp).

### 7.1. Cách 2 — Merge sort mỗi cửa sổ

In [6]:
def per_window_merge(nums, k):
    return min(inv_merge(nums[s:s + k])[1] for s in range(len(nums) - k + 1))

for a, k, exp in examples:
    print(per_window_merge(a, k), "kỳ vọng", exp)

0 kỳ vọng 0
6 kỳ vọng 6
0 kỳ vọng 0


### 7.2. Cách 3 — Cửa sổ trượt với list đã sắp (`insort`)

In [7]:
def sliding_insort(nums, k):
    win = []; inv = 0
    for x in nums[:k]:
        inv += len(win) - bisect_right(win, x)          # số phần tử > x
        insort(win, x)
    best = inv
    for i in range(k, len(nums)):
        y = nums[i - k]
        win.pop(bisect_left(win, y))                    # xoá một bản sao của y
        inv -= bisect_left(win, y)                      # số phần tử < y
        x = nums[i]
        inv += len(win) - bisect_right(win, x)
        insort(win, x)
        best = min(best, inv)
    return best

for a, k, exp in examples:
    print(sliding_insort(a, k), "kỳ vọng", exp)

0 kỳ vọng 0
6 kỳ vọng 6
0 kỳ vọng 0


### 7.3. Cách 4 — Cửa sổ trượt + Fenwick Tree (lời giải chính)

In [8]:
def sliding_bit(nums, k):
    rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
    r = [rank[v] for v in nums]
    bit = BIT(len(rank)); inv = 0
    for i in range(k):
        inv += i - bit.prefix(r[i])                     # i phần tử đã có, trừ số phần tử ≤ x
        bit.add(r[i], 1)
    best = inv
    for i in range(k, len(nums)):
        y = r[i - k]
        bit.add(y, -1)
        inv -= bit.prefix(y - 1)                        # số phần tử < y còn lại
        x = r[i]
        inv += (k - 1) - bit.prefix(x)                  # k-1 phần tử đang có, trừ số ≤ x
        bit.add(x, 1)
        best = min(best, inv)
    return best

for a, k, exp in examples:
    print(sliding_bit(a, k), "kỳ vọng", exp)

0 kỳ vọng 0
6 kỳ vọng 6
0 kỳ vọng 0


## 8. TRACE từng bước

In [9]:
def trace_sliding(nums, k):
    win = []; inv = 0
    for x in nums[:k]:
        g = len(win) - bisect_right(win, x)
        inv += g; insort(win, x)
        print(f"thêm {x}: +{g} (số phần tử lớn hơn) -> inv = {inv}")
    best = inv
    print(f"cửa sổ 0 {nums[:k]}: inv = {inv}")
    for i in range(k, len(nums)):
        y, x = nums[i - k], nums[i]
        win.pop(bisect_left(win, y)); l = bisect_left(win, y); inv -= l
        g = len(win) - bisect_right(win, x); inv += g; insort(win, x)
        best = min(best, inv)
        print(f"bỏ {y}: -{l}, thêm {x}: +{g} -> cửa sổ {nums[i-k+1:i+1]} inv = {inv} | best = {best}")
    return best

trace_sliding([3, 1, 2, 5, 4], 3)

thêm 3: +0 (số phần tử lớn hơn) -> inv = 0
thêm 1: +1 (số phần tử lớn hơn) -> inv = 1
thêm 2: +1 (số phần tử lớn hơn) -> inv = 2
cửa sổ 0 [3, 1, 2]: inv = 2
bỏ 3: -2, thêm 5: +0 -> cửa sổ [1, 2, 5] inv = 0 | best = 0
bỏ 1: -0, thêm 4: +1 -> cửa sổ [2, 5, 4] inv = 1 | best = 0


0

In [10]:
trace_sliding([4, 4, 2, 6, 1, 3], 4)
print("brute:", brute([4, 4, 2, 6, 1, 3], 4))

thêm 4: +0 (số phần tử lớn hơn) -> inv = 0
thêm 4: +0 (số phần tử lớn hơn) -> inv = 0
thêm 2: +2 (số phần tử lớn hơn) -> inv = 2
thêm 6: +0 (số phần tử lớn hơn) -> inv = 2
cửa sổ 0 [4, 4, 2, 6]: inv = 2
bỏ 4: -1, thêm 1: +3 -> cửa sổ [4, 2, 6, 1] inv = 4 | best = 2
bỏ 4: -2, thêm 3: +1 -> cửa sổ [2, 6, 1, 3] inv = 3 | best = 2
brute: 2


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def minInversionCount(self, nums: List[int], k: int) -> int:
        rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
        r = [rank[v] for v in nums]
        m = len(rank)
        c = [0] * (m + 1)

        def add(i, d):
            while i <= m:
                c[i] += d; i += i & -i

        def prefix(i):
            s = 0
            while i > 0:
                s += c[i]; i -= i & -i
            return s

        inv = 0
        for i in range(k):
            inv += i - prefix(r[i])
            add(r[i], 1)
        best = inv
        for i in range(k, len(nums)):
            y = r[i - k]
            add(y, -1)
            inv -= prefix(y - 1)
            x = r[i]
            inv += (k - 1) - prefix(x)
            add(x, 1)
            if inv < best:
                best = inv
        return best

sol = Solution()
for a, k, exp in examples:
    got = sol.minInversionCount(a, k)
    print(a, k, "->", got, "✔" if got == exp else "✘")

[3, 1, 2, 5, 4] 3 -> 0 ✔
[5, 3, 2, 1] 4 -> 6 ✔
[2, 1] 1 -> 0 ✔


## 10. Vì sao thuật toán đúng

**Bất biến:** sau khi xử lý cửa sổ `[s, s+k−1]`, BIT chứa đúng đa tập hạng của cửa sổ và `inv` = số nghịch thế của cửa sổ.

- **Dựng cửa sổ đầu:** khi thêm phần tử thứ `i` (0-indexed), nó đứng sau `i` phần tử đã có; các nghịch thế mới đúng là các cặp `(j, i)` với `nums[j] > nums[i]`, số lượng = `i − #{≤ nums[i]}`.
- **Bỏ phần tử trái `y`:** các nghịch thế chứa `y` là `(y, z)` với `z` sau `y` và `z < y`; mọi phần tử còn lại đều sau `y` ⇒ trừ `#{< y}`. Các cặp khác không đổi.
- **Thêm phần tử phải `x`:** tương tự, cộng `#{> x}` trong `k − 1` phần tử còn lại.
- So sánh nghiêm ngặt (`<`, `>`) qua `prefix(y−1)` và `prefix(x)` ⇒ phần tử bằng nhau không bị tính.

Lấy `min` trên mọi cửa sổ ⇒ đáp án.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O((n−k+1) · k²)` | `O(k)` |
| 2. Merge sort mỗi cửa sổ | `O((n−k+1) · k log k)` | `O(k)` |
| 3. `insort` trượt | `O(n · k)` (memmove nhanh) | `O(k)` |
| 4. BIT trượt | `O(n log n)` | `O(n)` |

## 12. Bộ test

In [12]:
METHODS = [brute, per_window_merge, sliding_insort, sliding_bit, lambda a, k: Solution().minInversionCount(a, k)]
for a, k, exp in examples:
    assert all(f(a, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên
assert Solution().minInversionCount([7], 1) == 0
assert Solution().minInversionCount([2, 2, 2, 2], 3) == 0                 # bằng nhau không phải nghịch thế
assert Solution().minInversionCount([5, 4, 3, 2, 1], 5) == 10
assert Solution().minInversionCount([1, 3, 2, 3, 1], 2) == 0
print("Ca biên: OK")

Ca biên: OK


In [14]:
# 12.3 ngẫu nhiên: mọi cách phải khớp vét cạn (giá trị nhỏ để có nhiều phần tử trùng)
rng = random.Random(3768)
for _ in range(1500):
    n = rng.randint(1, 12); k = rng.randint(1, n)
    a = [rng.randint(1, 5) for _ in range(n)]
    b = brute(a, k)
    assert all(f(a, k) == b for f in METHODS[1:]), (a, k)
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [15]:
# 12.4 test lớn
rng = random.Random(7)
a = [rng.randint(1, 10**9) for _ in range(10**5)]
for k in (2, 50000, 10**5):
    t0 = time.time(); x = Solution().minInversionCount(a, k); t1 = time.time()
    print(f"k = {k}: đáp án {x}, thời gian {t1 - t0:.2f}s, khớp insort: {x == sliding_insort(a, k)}")

k = 2: đáp án 0, thời gian 0.36s, khớp insort: True


k = 50000: đáp án 622828570, thời gian 0.32s, khớp insort: True


k = 100000: đáp án 2498997096, thời gian 0.23s, khớp insort: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khi bỏ phần tử trái lại trừ số phần tử **lớn hơn** nó

Phần tử trái đứng **trước** mọi phần tử khác; nó tạo nghịch thế với các phần tử **nhỏ hơn** đứng sau.

In [16]:
def wrong_remove(nums, k):
    win = []; inv = 0
    for x in nums[:k]:
        inv += len(win) - bisect_right(win, x); insort(win, x)
    best = inv
    for i in range(k, len(nums)):
        y = nums[i - k]; win.pop(bisect_left(win, y))
        inv -= len(win) - bisect_right(win, y)               # SAI: phải là số phần tử < y
        x = nums[i]; inv += len(win) - bisect_right(win, x); insort(win, x)
        best = min(best, inv)
    return best

print("sai:", wrong_remove([3, 1, 2, 5, 4], 3), "| đúng:", brute([3, 1, 2, 5, 4], 3))

sai: 1 | đúng: 0


### Lỗi 2 — Tính cả cặp bằng nhau là nghịch thế (`≥` thay vì `>`)

In [17]:
def wrong_equal(nums, k):
    win = []; inv = 0
    for x in nums[:k]:
        inv += len(win) - bisect_left(win, x); insort(win, x)          # SAI: đếm cả = x
    best = inv
    for i in range(k, len(nums)):
        y = nums[i - k]; win.pop(bisect_left(win, y)); inv -= bisect_right(win, y)
        x = nums[i]; inv += len(win) - bisect_left(win, x); insort(win, x)
        best = min(best, inv)
    return best

print("sai:", wrong_equal([2, 2, 2, 2], 3), "| đúng:", brute([2, 2, 2, 2], 3))

sai: 3 | đúng: 0


### Lỗi 3 — Dùng BIT kích thước `max(nums)` (không nén toạ độ)

`max(nums)` tới `10^9` ⇒ mảng BIT `10^9` phần tử — tràn bộ nhớ. Luôn nén về hạng `1..m`.

In [18]:
print("bộ nhớ list 10^9 số nguyên Python (ước lượng, GB):", 10**9 * 8 / 1e9, "+ đối tượng int")

bộ nhớ list 10^9 số nguyên Python (ước lượng, GB): 8.0 + đối tượng int


### Lỗi 4 — Thứ tự cập nhật BIT khi trượt

Nếu thêm `x` vào BIT **trước** khi đếm `#{> x}`, `x` không tự đếm chính nó (vì `>`), nhưng nếu dùng công thức `size − prefix(x)` với `size` đã tăng thì sẽ sai lệch 1.

In [19]:
def wrong_order(nums, k):
    rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}; r = [rank[v] for v in nums]
    bit = BIT(len(rank)); inv = 0
    for i in range(k):
        bit.add(r[i], 1); inv += (i + 1) - bit.prefix(r[i]) + 0      # đúng vì size = i+1 sau khi thêm
    best = inv
    for i in range(k, len(nums)):
        y = r[i - k]; inv -= bit.prefix(y - 1); bit.add(y, -1)
        x = r[i]; bit.add(x, 1)
        inv += k - bit.prefix(x) + 1                                  # SAI: lệch 1
        best = min(best, inv)
    return best

print("sai:", wrong_order([3, 1, 2, 5, 4], 3), "| đúng:", brute([3, 1, 2, 5, 4], 3))

sai: 1 | đúng: 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nghịch thế trong **mọi** cửa sổ độ dài `k`" ⇒ cửa sổ trượt: chỉ phần tử ở **hai đầu** thay đổi.
- Phần tử ở đầu trái/phải so với **toàn bộ** cửa sổ ⇒ đếm "nhỏ hơn/lớn hơn" bằng BIT hoặc cấu trúc có thứ tự.
- Giá trị lớn ⇒ nén toạ độ.
- Trong Python, `bisect.insort` thường đủ nhanh cho `n = 10^5` (memmove), nhưng BIT có chặn trên an toàn.

## 15. Biến thể của bài

1. **Tối đa** số nghịch thế thay vì tối thiểu: cùng thuật toán, đổi `min` thành `max`.
2. **Tổng nghịch thế của mọi cửa sổ**.
3. **Nghịch thế của mọi mảng con** (không cố định độ dài) — đếm đóng góp mỗi cặp: `(i+1)·(n−j)`.
4. **Truy vấn nghịch thế đoạn bất kỳ** (offline): thuật toán Mo + BIT.

In [20]:
# Biến thể 3: tổng nghịch thế trên mọi mảng con = Σ_{i<j, a_i>a_j} (i+1)(n-j)
def total_all_subarrays(a):
    n = len(a); rank = {v: i + 1 for i, v in enumerate(sorted(set(a)))}
    bit = BIT(len(rank)); total = 0; seen_weight = 0
    for j, v in enumerate(a):
        # tổng (i+1) của các i < j có a_i > a_j = seen_weight - tổng (i+1) với a_i <= a_j
        total += (seen_weight - bit.prefix(rank[v])) * (n - j)
        bit.add(rank[v], j + 1); seen_weight += j + 1
    return total

def total_brute(a):
    n = len(a)
    return sum(brute(a[s:e], e - s) for s in range(n) for e in range(s + 1, n + 1))

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 9))]
    assert total_all_subarrays(a) == total_brute(a)
print("Biến thể 3 OK; ví dụ:", total_all_subarrays([3, 1, 2, 5, 4]))

Biến thể 3 OK; ví dụ: 11


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Đếm từng cặp | `O(n k²)` | `O(k)` | Kiểm chứng |
| 2. Merge sort | Đếm mỗi cửa sổ | `O(n k log k)` | `O(k)` | `k` nhỏ |
| 3. `insort` | Cửa sổ trượt, list sắp | `O(n k)` | `O(k)` | Python, nhanh trong thực tế |
| 4. BIT | Cửa sổ trượt, cây Fenwick | `O(n log n)` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minInversionCount([4,3,2,1,5], 2)` = ?

**Bài 2 (tính tay).** Trace `inv` cho `[2,4,1,3]`, `k = 3`.

**Bài 3 (điền TODO).** Viết `max_inversion_count(nums, k)`.

**Bài 4 (tự viết lại).** Viết lời giải dùng Segment Tree đếm thay cho BIT.

**Bài 5 (mini challenge).** Trả về **chỉ số bắt đầu nhỏ nhất** của cửa sổ đạt tối thiểu.

In [21]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def max_inversion_count(nums, k):
    # TODO
    pass

In [22]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def argmin_window(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cửa sổ `[1,5]` có 0. (2) Các cửa sổ khác đều 1. (3) Đáp án `0`.

**Bài 2.** (1) `[2,4,1]`: thêm 2 (+0), 4 (+0), 1 (+2) ⇒ 2. (2) Bỏ 2 (−1 vì có 1 < 2), thêm 3 (+1 vì 4 > 3) ⇒ 2. (3) Đáp án `2`.

**Bài 3.** (1) Copy lời giải. (2) `best = max(...)`. (3) Đối chiếu vét cạn.

**Bài 4.** (1) Cây tổng trên hạng. (2) `query(1, x−1)` cho "nhỏ hơn". (3) `query(x+1, m)` cho "lớn hơn".

**Bài 5.** (1) Ghi `inv` từng cửa sổ. (2) Chỉ cập nhật khi `inv < best` (nghiêm ngặt). (3) Trả `s`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().minInversionCount([4, 3, 2, 1, 5], 2) == 0 == brute([4, 3, 2, 1, 5], 2)
assert Solution().minInversionCount([2, 4, 1, 3], 3) == 2 == brute([2, 4, 1, 3], 3)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [24]:
# Bài 3, 5
def window_invs(nums, k):
    win = []; inv = 0; out = []
    for x in nums[:k]:
        inv += len(win) - bisect_right(win, x); insort(win, x)
    out.append(inv)
    for i in range(k, len(nums)):
        y = nums[i - k]; win.pop(bisect_left(win, y)); inv -= bisect_left(win, y)
        x = nums[i]; inv += len(win) - bisect_right(win, x); insort(win, x); out.append(inv)
    return out

def max_inversion_count_sol(nums, k):
    return max(window_invs(nums, k))

def argmin_window_sol(nums, k):
    w = window_invs(nums, k); return w.index(min(w))

rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 10); k = rng.randint(1, n); a = [rng.randint(1, 5) for _ in range(n)]
    per = [brute(a[s:s + k], k) for s in range(n - k + 1)]
    assert max_inversion_count_sol(a, k) == max(per) and argmin_window_sol(a, k) == per.index(min(per))
print("Bài 3, 5 OK")

Bài 3, 5 OK


In [25]:
# Bài 4: Segment Tree đếm
def sliding_segtree(nums, k):
    rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}; r = [rank[v] for v in nums]
    m = len(rank); size = 1
    while size < m + 2: size *= 2
    t = [0] * (2 * size)
    def add(i, d):
        i += size; t[i] += d; i //= 2
        while i: t[i] = t[2 * i] + t[2 * i + 1]; i //= 2
    def query(lo, hi):                       # tổng [lo, hi]
        if lo > hi: return 0
        res = 0; lo += size; hi += size + 1
        while lo < hi:
            if lo & 1: res += t[lo]; lo += 1
            if hi & 1: hi -= 1; res += t[hi]
            lo //= 2; hi //= 2
        return res
    inv = 0
    for i in range(k):
        inv += query(r[i] + 1, m); add(r[i], 1)
    best = inv
    for i in range(k, len(nums)):
        y = r[i - k]; add(y, -1); inv -= query(1, y - 1)
        x = r[i]; inv += query(x + 1, m); add(x, 1); best = min(best, inv)
    return best

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 10); k = rng.randint(1, n); a = [rng.randint(1, 5) for _ in range(n)]
    assert sliding_segtree(a, k) == brute(a, k)
print("Bài 4 OK")

Bài 4 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
rank = {v: i+1 for i, v in enumerate(sorted(set(nums)))}
# cửa sổ đầu: inv += (#đã có) - prefix(rank[x]); add(rank[x])
# trượt:     add(y, -1); inv -= prefix(y - 1)          # y rời trái: trừ số phần tử < y
#            inv += (k - 1) - prefix(x); add(x, +1)    # x vào phải: cộng số phần tử > x
best = min(best, inv)
```

**Checklist:**

- [ ] Nén toạ độ trước khi dùng BIT?
- [ ] Bỏ trái: trừ **nhỏ hơn**; thêm phải: cộng **lớn hơn**?
- [ ] So sánh nghiêm ngặt (bằng nhau không tính)?
- [ ] Thứ tự `add`/`prefix` khớp với công thức kích thước?
- [ ] `k = 1` ⇒ 0; `k = n` ⇒ một cửa sổ?